# EA1 - Actividad 1.3: DataFrames en Spark

## Objetivos
- Dominar operaciones de DataFrame: select, filter, groupBy, join
- Utilizar funciones de agregacion y funciones de columna
- Trabajar con funciones de fecha
- Crear columnas derivadas con `withColumn`, `when/otherwise`

## Conceptos Clave

### DataFrames vs RDDs

| Aspecto | RDD | DataFrame |
|---------|-----|-----------|
| **Esquema** | Sin esquema (datos no estructurados) | Con esquema (columnas tipadas) |
| **Optimizacion** | Sin optimizacion automatica | Optimizador **Catalyst** + motor **Tungsten** |
| **API** | Funcional (map, filter, reduce) | Declarativa (select, filter, groupBy) - similar a SQL |
| **Rendimiento** | Menor (sin optimizaciones) | Mayor (plan de ejecucion optimizado) |
| **Uso recomendado** | Datos no estructurados, control fino | Datos estructurados/semi-estructurados |

**Recomendacion:** Usar DataFrames siempre que sea posible. Los RDDs son utiles cuando necesitas control a bajo nivel o trabajas con datos no estructurados.

In [ ]:
# Setup: Crear SparkSession e importar funciones
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("03_dataframes_intro") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## Cargar los datos

Trabajaremos con dos datasets:
- **sales.csv:** Datos de ventas semanales por tienda y departamento
- **stores.csv:** Informacion de cada tienda (tipo y tamano)

In [ ]:
# Leer los datasets
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

In [ ]:
# Explorar df_sales
print("=== SALES ===")
df_sales.show(5)
df_sales.printSchema()
print(f"Filas: {df_sales.count()}, Columnas: {len(df_sales.columns)}")

In [ ]:
# Explorar df_stores
print("=== STORES ===")
df_stores.show(5)
df_stores.printSchema()
print(f"Filas: {df_stores.count()}, Columnas: {len(df_stores.columns)}")

## `select()` y `withColumn()`

- **`select()`**: Elige columnas especificas (como SELECT en SQL)
- **`withColumn()`**: Agrega o reemplaza una columna

In [ ]:
# select: elegir columnas especificas
df_sales.select("Store", "Weekly_Sales").show(5)

# withColumn: crear una nueva columna
# Ejemplo: convertir ventas a una columna explicita en USD
df_con_usd = df_sales.withColumn("Sales_USD", F.col("Weekly_Sales") * 1.0)
df_con_usd.select("Store", "Weekly_Sales", "Sales_USD").show(5)

## `filter()` con condiciones multiples

Se pueden combinar condiciones usando `&` (AND) y `|` (OR). Cada condicion debe ir entre parentesis.

In [ ]:
# Filtrar: ventas mayores a 50000 en dias festivos
df_filtrado = df_sales.filter(
    (F.col("Weekly_Sales") > 50000) & (F.col("IsHoliday") == True)
)

print(f"Registros con ventas > 50K en festivos: {df_filtrado.count()}")
df_filtrado.show(5)

## `groupBy()` + `agg()`

Agrupar datos y aplicar funciones de agregacion: `sum`, `avg`, `count`, `min`, `max`, etc.

In [ ]:
# Agrupar por tienda y calcular total y promedio de ventas
df_por_tienda = df_sales.groupBy("Store").agg(
    F.sum("Weekly_Sales").alias("total_ventas"),
    F.avg("Weekly_Sales").alias("promedio_ventas"),
    F.count("Weekly_Sales").alias("num_registros")
)

df_por_tienda.orderBy("total_ventas", ascending=False).show(10)

## `join()` - Unir DataFrames

El join permite combinar dos DataFrames basandose en una columna en comun, similar a JOIN en SQL.

In [ ]:
# Join: unir ventas con informacion de tiendas
df_joined = df_sales.join(df_stores, "Store")

print(f"Columnas despues del join: {df_joined.columns}")
df_joined.show(5)

## Funciones de Fecha

Spark incluye funciones para trabajar con columnas de fecha: parsear, extraer componentes, calcular diferencias, etc.

In [ ]:
# Convertir la columna Date a tipo fecha (si no lo es ya)
df_con_fecha = df_sales.withColumn("Fecha", F.to_date(F.col("Date"), "yyyy-MM-dd"))

# Extraer componentes de la fecha
df_con_fecha = df_con_fecha \
    .withColumn("Anio", F.year("Fecha")) \
    .withColumn("Mes", F.month("Fecha")) \
    .withColumn("Dia_Semana", F.dayofweek("Fecha"))

df_con_fecha.select("Store", "Date", "Fecha", "Anio", "Mes", "Dia_Semana").show(5)

## `when()` / `otherwise()` - Columnas condicionales

Permite crear columnas basadas en condiciones, similar a CASE WHEN en SQL.

In [ ]:
# Crear categoria de ventas basada en el monto
df_categorizado = df_sales.withColumn(
    "categoria",
    F.when(F.col("Weekly_Sales") > 50000, "Alta")
     .when(F.col("Weekly_Sales") > 20000, "Media")
     .otherwise("Baja")
)

df_categorizado.select("Store", "Weekly_Sales", "categoria").show(10)

# Contar cuantos registros hay en cada categoria
df_categorizado.groupBy("categoria").count().show()

---
## Ejercicios

Ahora es tu turno de practicar con DataFrames.

In [ ]:
# =============================================================
# EJERCICIO 1: Ventas totales por tipo de tienda
# =============================================================
# Join entre ventas y tiendas, luego agrupar por tipo

df_joined = df_sales.join(df_stores, "Store")
df_ventas_tipo = df_joined.groupBy("Type") \
    .agg(F.sum("Weekly_Sales").alias("total_ventas")) \
    .orderBy("total_ventas", ascending=False)
df_ventas_tipo.show()

> **Nota docente:** Este ejercicio combina dos operaciones clave: `join` y `groupBy + agg`. Al usar `join(df_stores, "Store")` con un solo string, Spark hace un inner join automatico y evita duplicar la columna "Store". Si se usara `join(df_stores, df_sales.Store == df_stores.Store)`, la columna apareceria duplicada. Error comun: olvidar hacer el join antes de agrupar (la columna "Type" esta en df_stores, no en df_sales). Usar `.alias()` permite dar nombres descriptivos a las columnas agregadas, lo cual es buena practica.

In [ ]:
# =============================================================
# EJERCICIO 2: Tienda con mayor venta promedio semanal
# =============================================================
# Calcular el promedio de ventas por tienda y encontrar la mayor

df_promedio = df_sales.groupBy("Store") \
    .agg(F.avg("Weekly_Sales").alias("promedio_semanal")) \
    .orderBy("promedio_semanal", ascending=False)
df_promedio.show(1)

> **Nota docente:** `F.avg()` calcula el promedio aritmetico. `.show(1)` muestra solo la primera fila, que al estar ordenada descendentemente corresponde a la tienda con mayor promedio. Alternativa: se podria usar `df_promedio.first()` para obtener la Row directamente. Error comun: confundir `F.avg()` con `F.mean()` (ambos son equivalentes en Spark). Tambien es frecuente que los estudiantes olviden el `alias()` y luego no sepan como referenciar la columna calculada.

In [ ]:
# =============================================================
# EJERCICIO 3: Crear columna "temporada" basada en el mes
# =============================================================
# Usar estaciones del hemisferio sur (Chile)

from pyspark.sql.functions import month, col, when

df_con_mes = df_sales.withColumn("mes", month(F.to_date("Date", "yyyy-MM-dd")))

df_temporada = df_con_mes.withColumn("temporada",
    when(col("mes").isin(12, 1, 2), "Verano")
    .when(col("mes").isin(3, 4, 5), "Otono")
    .when(col("mes").isin(6, 7, 8), "Invierno")
    .otherwise("Primavera")
)

# Mostrar ejemplos
df_temporada.select("Store", "Date", "mes", "temporada", "Weekly_Sales").show(10)

# Contar registros por temporada
df_temporada.groupBy("temporada").count().show()

> **Nota docente:** Este ejercicio combina funciones de fecha con columnas condicionales. Puntos clave: (1) `F.to_date()` requiere el formato correcto del string de entrada; (2) `when().when().otherwise()` es analogo a `CASE WHEN ... WHEN ... ELSE` en SQL; (3) `.isin()` es mas limpio que multiples condiciones con `|`. Error comun: los estudiantes del hemisferio norte pueden confundir las estaciones (en Chile, diciembre es verano). Otro error frecuente: olvidar que `when()` debe encadenarse directamente (sin `F.` adicional en el segundo `when`).

---
## Resumen

En esta actividad aprendimos:

1. **DataFrames vs RDDs:** Los DataFrames tienen esquema y se benefician del optimizador Catalyst
2. **select() y withColumn():** Seleccionar y crear columnas
3. **filter():** Filtrar filas con condiciones simples y compuestas
4. **groupBy() + agg():** Agrupar datos y calcular agregaciones (sum, avg, count, etc.)
5. **join():** Unir DataFrames por columnas en comun
6. **Funciones de fecha:** to_date, year, month, dayofweek
7. **when/otherwise:** Crear columnas condicionales

---
## Desafio Extra (Opcional)

**Crecimiento porcentual de ventas mes a mes por tienda**

Usa **Window Functions** para calcular el crecimiento porcentual de ventas de un mes al siguiente para cada tienda.

Pistas:
- Primero agrega ventas por tienda y mes
- Usa `from pyspark.sql.window import Window`
- Define una ventana: `Window.partitionBy("Store").orderBy("Mes")`
- Usa `F.lag("total_ventas", 1)` para obtener el valor del mes anterior
- Calcula: `((actual - anterior) / anterior) * 100`

In [ ]:
# =============================================================
# DESAFIO: Crecimiento porcentual mes a mes por tienda
# =============================================================
# Usar Window Functions para calcular crecimiento de ventas

from pyspark.sql.window import Window

# Paso 1: Agregar ventas por tienda y mes
df_mensual = df_sales.withColumn("mes", F.month(F.to_date("Date", "yyyy-MM-dd"))) \
    .groupBy("Store", "mes") \
    .agg(F.sum("Weekly_Sales").alias("ventas_mes"))

# Paso 2: Definir ventana particionada por tienda, ordenada por mes
w = Window.partitionBy("Store").orderBy("mes")

# Paso 3: Calcular ventas del mes anterior con lag()
# Paso 4: Calcular porcentaje de crecimiento
df_crecimiento = df_mensual.withColumn("ventas_anterior", F.lag("ventas_mes").over(w)) \
    .withColumn("crecimiento_pct",
        F.when(F.col("ventas_anterior").isNotNull(),
            ((F.col("ventas_mes") - F.col("ventas_anterior")) / F.col("ventas_anterior") * 100)
        ).otherwise(None)
    )

# Paso 5: Mostrar resultados
df_crecimiento.orderBy("Store", "mes").show(20)

> **Nota docente:** Las Window Functions son un concepto avanzado pero muy poderoso, equivalente a las window functions de SQL. Puntos clave: (1) `Window.partitionBy("Store")` crea una ventana independiente para cada tienda; (2) `orderBy("mes")` establece el orden dentro de cada ventana; (3) `F.lag("col", 1)` obtiene el valor de la fila anterior (el `1` es por defecto y se puede omitir); (4) el primer mes de cada tienda tendra `null` como ventas_anterior, de ahi el `when().otherwise(None)` para evitar division por cero. Las Window Functions NO agregan filas como `groupBy`, sino que agregan columnas calculadas basadas en ventanas. Este concepto se profundizara en EA2.

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")